# Citation Index API — interactive client test

Run this notebook from the project root with the package installed:

```bash
pip install -e ".[dev]"
pip install ipykernel   # if needed
python -m ipykernel install --user --name citation-index-api-client
```

Select that kernel in Jupyter / VS Code, then run the cells top to bottom.

In [1]:
from pathlib import Path

from citation_index_api_client import CitationIndexClient
from citation_index_api_client.exceptions import APIError, JobFailedError, JobTimeoutError

# --- configure ---
BASE_URL = "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl"
PDF_PATH = Path("paper.pdf")  # set to your PDF, or None to skip text extraction
POLL_INTERVAL = 2.0
JOB_TIMEOUT = 600.0

client = CitationIndexClient(base_url=BASE_URL, timeout=120.0)
print(f"Client → {client.base_url}")

Client → https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl


## Health & API info

In [2]:
try:
    print("health:", client.health())
    print("root:", client.root())
except APIError as e:
    print(f"API error {e.status_code}: {e.detail}")

health: {'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '0.3.0'}
root: {'name': 'Citation Index API', 'version': '0.3.0', 'docs': '/docs', 'health': '/health'}


## Extract text from a PDF

Set `PDF_PATH` in the first cell to a local `.pdf` file. This enqueues an async job and waits for the result.

In [3]:
PDF_PATH = Path("../test_data/reg_0035-2039_1989_num_102_485_2445.pdf")

In [4]:
if PDF_PATH is None or not PDF_PATH.exists():
    print(f"Skipping text extraction — set PDF_PATH to an existing file (got {PDF_PATH!r})")
else:
    job = client.extract_text(PDF_PATH, extractor="pymupdf", markdown=True)
    #job = client.extract_text(PDF_PATH, extractor="mineru", markdown=True)
    print(f"Enqueued: {job}")

    status = client.wait_for_job(
        job.job_id, 
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    print(f"Finished: {status}")

    text_result = client.get_job_result(job.job_id)
    if isinstance(text_result, dict):
        preview = str(text_result)[:500]
    else:
        preview = str(text_result)[:500]
    print(f"Result preview:\n{preview}...")

Enqueued: JobResponse(job_id='bfe46da8-a6ce-4315-bd23-e0ef20841f02', status='queued', created_at='2026-09-08T14:34:19.439578', message='Text extraction job enqueued')
Finished: JobStatus(job_id='bfe46da8-a6ce-4315-bd23-e0ef20841f02', status='completed', created_at='2026-09-08T14:34:19.439578', started_at='2026-09-08T14:34:19.507939', completed_at='2026-09-08T14:34:27.891404', current_stage='text_extraction', completed_stages=['text_extraction'], error=None, progress=None)
Result preview:
{'text': "Revue des Études Grecques \n\n\n\nSur le troisième stasimon d'Œdipe-Roi Albert Machin \n\nCiter ce document / Cite this document : \n\nMachin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvierjuin 1989. pp. 192-201 ; \n\ndoi : https://doi.org/10.3406/reg.1989.2445 \n\nhttps://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445 \n\nFichier pdf généré le 18/04/2018 \n\n\n\n# SUR LE TROISIEME STASIMON O'ŒDIPE-ROI \n\n\n\n{O...


In [7]:
print(text_result['text'][:500])

Revue des Études Grecques 



Sur le troisième stasimon d'Œdipe-Roi Albert Machin 

Citer ce document / Cite this document : 

Machin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvierjuin 1989. pp. 192-201 ; 

doi : https://doi.org/10.3406/reg.1989.2445 

https://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445 

Fichier pdf généré le 18/04/2018 



# SUR LE TROISIEME STASIMON O'ŒDIPE-ROI 



{O.R. vientLe de 1086-1 chant prof


## Extract references from markdown

Uses a small sample bibliography block. Replace `SAMPLE_MARKDOWN` with output from text extraction if you have it.

In [8]:
#with open("../test_data/Holanda-Brasliadailyinvention-1989.md", "r") as f:
#    text_result = {}
#    text_result['text'] = f.read()

In [9]:
job = client.extract_references(
    text_result['text'],
    method="full_text",
    temperature=0.3,
)
print(f"Enqueued: {job}")

try:
    refs_result = client.wait_for_result(
        job.job_id,
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    refs_result
except JobFailedError as e:
    print(f"Job failed: {e}")
except JobTimeoutError as e:
    print(f"Timed out after {e.timeout}s")

Enqueued: JobResponse(job_id='eacd9f5b-0f27-40f9-8062-e00a61e46536', status='queued', created_at='2026-09-08T14:35:15.234644', message='Reference extraction job enqueued')


In [10]:
refs_result

{'references': ['R. Dawe, éd. Soph. (Teubner), 1975.',
  'P. Mazon-A. Dain, éd. Soph., Belles-Lettres, 1958.',
  'C. M. Bowra, Sophoclean Tragedy, Oxford, 1944.',
  'A. Lesky, Greek Tragedy, éd. angl. Londres, 1965.',
  'R. C. Jebb, éd. O.T., Cambridge, 1914.',
  'J. C. Kamerbeek, éd. O.T., Leyde, 1967.',
  "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980.",
  'H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970.',
  'W. Kranz, Stasimon, Berlin, 1933.',
  'G. M. Kirkwood, A Study of Sophoclean Drama, New York, 1958.',
  'T. B. L. Webster, An Introduction to Sophocles, Oxford, 1936.',
  'M. A. Bayfield, éd. Antigone, N-Y, 1968.',
  'G. Mûller, éd. Antigone, Leyde, 1967.',
  'A. M. Dale, «Stasimon and Hyporchème», Eranos, 48, 1950.',
  'A. M. Dale, The Lyric Metres in Greek Drama, 2e éd., Cambridge, 1968.',
  'P. E. Easterling, éd. Trach., Cambridge, 1982.',
  'T. von Wilamowitz, die dramatische Technik des Sophokles, Berlin, 1917.',
  'R. C. Jebb, éd. Phil., Cambridge

## Parse reference strings

Send raw bibliography lines to the parser endpoint.

In [ ]:
#job = client.parse_references(refs_result['references'], parser="grobid", temperature=0.0)
#job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0, prompt_name="prompts/reference_parsing.md")
#job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0, prompt_name="prompts/reference_parsing_linkedbook.md")
job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0)
print(f"Enqueued: {job}")

parsed_result = client.wait_for_result(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
parsed_result

Enqueued: JobResponse(job_id='afaa0e55-99a5-4f53-83af-9771b56cda59', status='queued', created_at='2026-07-31T05:47:48.387565', message='Reference parsing job enqueued')


{'references': [{'full_title': None,
   'journal_title': None,
   'authors': [{'first_name': 'Albert',
     'middle_name': None,
     'surname': 'Machin',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': None,
   'translator': None,
   'publication_place': None,
   'publication_year': None,
   'publication_date_raw': 'Janvier-juin 1989',
   'identifiers': [],
   'ref_type': None,
   'raw': {},
   'volume': None,
   'issue': '485-486',
   'pages': '192-201',
   'cited_range': None,
   'footnote_number': '1'},
  {'full_title': None,
   'journal_title': None,
   'authors': [{'first_name': 'R.',
     'middle_name': None,
     'surname': 'Dawe',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': None,
   'translator': None,
   'publication_place': None,
   'publication_year': None,
   'publication_date_raw': '1975',
   'identifiers': [],
   'ref_type': None,
   'raw': {},
   'volume': None,
   'issue': None,
   'pages':

In [18]:
len(parsed_result['references'])

56

## Link references to external indexes

`link_references` submits the raw reference strings extracted above to
`POST /link/references`, which resolves each one against OpenAlex, Matilda and
Wikidata. Pass a list to submit a batch, or a single string for one reference.

Each target returns an `id` and a `doi`, both `None` when no confident match is
found — the service requires a DOI hit or a near-exact title match. Linking hits
live external APIs once per reference per target, so keep `LINKING_LIMIT` small.

In [ ]:
LINKING_TARGETS = "all"   # or "openalex", or "openalex,wikidata"
LINKING_LIMIT = 5         # bounded: each reference hits every target's live API

references_to_link = refs_result['references'][:LINKING_LIMIT]

job = client.link_references(references_to_link, target=LINKING_TARGETS)
print(f"Enqueued: {job}")

linked = client.wait_for_links(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
print(f"Linked {len(linked)} references")

In [ ]:
for item in linked:
    print(f"\n{item.reference[:90]}")
    for target, link in item.links.items():
        mark = "✓" if link.is_linked else "·"
        print(f"  {mark} {target:<9} id={link.id or '—'}  doi={link.doi or '—'}")

hits = sum(link.is_linked for item in linked for link in item.links.values())
lookups = sum(len(item.links) for item in linked)
print(f"\n{hits} / {lookups} target lookups matched")

## Manual job polling (optional)

Use this pattern when you want to inspect status between polls instead of blocking on `wait_for_result`.

In [11]:
# Re-use the last job_id from the cell above, or paste one here:
job_id = job.job_id

status = client.get_job_status(job_id)
print(status)
print("terminal?", status.is_terminal)

if status.status == "completed":
    result = client.get_job_result(job_id)
    print(result)

JobStatus(job_id='afaa0e55-99a5-4f53-83af-9771b56cda59', status='completed', created_at='2026-07-31T05:47:48.387565', started_at=None, completed_at='2026-07-31T05:48:58.514835', current_stage='reference_parsing', completed_stages=['reference_parsing'], error=None, progress=None)
terminal? True
{'references': [{'full_title': None, 'journal_title': None, 'authors': [{'first_name': 'Albert', 'middle_name': None, 'surname': 'Machin', 'name_link': None, 'role_name': None}], 'editors': None, 'publisher': None, 'translator': None, 'publication_place': None, 'publication_year': None, 'publication_date_raw': 'Janvier-juin 1989', 'identifiers': [], 'ref_type': None, 'raw': {}, 'volume': None, 'issue': '485-486', 'pages': '192-201', 'cited_range': None, 'footnote_number': '1'}, {'full_title': None, 'journal_title': None, 'authors': [{'first_name': 'R.', 'middle_name': None, 'surname': 'Dawe', 'name_link': None, 'role_name': None}], 'editors': None, 'publisher': None, 'translator': None, 'publicat

In [12]:
client.close()
print("Client closed.")

Client closed.
